In [ ]:
from mpramnist.Vaishnav2024 import VaishnavDataset
from mpramnist.Vaishnav2024 import LitModel_Vaishnav

import mpramnist.transforms as t

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1024
NUM_WORKERS = 103

length = 110
plasmid = VaishnavDataset.PLASMID.upper()
insert_start = plasmid.find("N" * 80)
right_flank = VaishnavDataset.RIGHT_FLANK
left_flank = plasmid[insert_start - length : insert_start]

**Important note**: Sequence lengths vary. To standardize them:

* Original flanks should be preserved

* Missing regions need to be supplemented from the source plasmid, use LeftFlank for it

* All sequences must be adjusted to the default 110 bp length (as in the original protocol)

In [5]:
# preprocessing
train_transform = t.Compose(
    [
        t.AddFlanks(left_flank, right_flank),
        t.LeftCrop(length, length),
        t.ReverseComplement(0.5),
        t.Seq2Tensor(),
    ]
)
val_test_transform = t.Compose(
    [
        t.AddFlanks(left_flank, right_flank),
        t.LeftCrop(length, length),
        t.ReverseComplement(0),
        t.Seq2Tensor(),
    ]
)

In the original study, two complementary environments with opposing selective pressures on URA3 gene expression (encoding an enzyme responsible for uracil synthesis) were investigated:

`defined` environment, where organismal fitness increases with gene expression (up to saturation);

`complex` environment + 5-FOA, where fitness decreases with Ura3p expression.

Use the `dataset_env_type` parameter to select either `'defined'` or `'complex'`.

# Dataset Specifications:

`defined`: (1) Contains 20 million sequences (2) 10% allocated for validation (3) Remainder used for training

`complex`: (1) Contains 31 million sequences (2) 10% allocated for validation (3) Remainder used for training

# Train **defined** env type

In [ ]:
# load the data
train_dataset = VaishnavDataset(split="train",dataset_env_type="defined",transform=train_transform,root="../data/",)
val_dataset = VaishnavDataset(split="val",dataset_env_type="defined",transform=val_test_transform,root="../data/",)

print(len(train_dataset), len(val_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

In [ ]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[1, 1, 1, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_defined = LitModel_Vaishnav(model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

# Train the model
trainer.fit(seq_model_defined, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
def meaned_prediction(forw, rev, trainer, seq_model, name, is_paired=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print(name + " Pearson correlation")

    if is_paired:
        y_preds_forw_alt = torch.cat(
            [pred["alt_predicted"] for pred in predictions_forw]
        )
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets)

    return pears(mean_forw, targets)

## Test Sequences:

Test sequences are divided into three categories per environment:

* Reference (`native`)

* Alternative (`drift`)

* Paired (`paired`)

Use the `test_dataset_type`

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="native",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="native",transform=rev_transform,root="../data/",)

print("native info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="native")

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="drift",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="drift",transform=rev_transform,root="../data/",)

print("drift info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="drift")

### Note on paired sequences:

Each paired sequence contains: (1) A reference sequence (2) An alternative sequence (3) Differential expression column

In [17]:
dataset_paired = VaishnavDataset(
    split="test",
    dataset_env_type="defined",
    test_dataset_type="paired",
    root="../data/",
)
dataset_paired[0]

({'seq': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT',
  'seq_alt': 'CTTTAAATTCGGTGGGGACGCGTCGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT'},
 tensor(0.6423))

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="paired",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="defined",test_dataset_type="paired",transform=rev_transform,root="../data/",)

print("paired info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="paired")

# Train **complex** env type

In [ ]:
# load the data
train_dataset = VaishnavDataset(split="train",dataset_env_type="complex",transform=train_transform,root="../data/",)
val_dataset = VaishnavDataset(split="val",dataset_env_type="complex",transform=val_test_transform,root="../data/",)

print(len(train_dataset), len(val_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

In [ ]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[1, 1, 1, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model_complex = LitModel_Vaishnav(
    model=model, loss=nn.MSELoss(), weight_decay=1e-1, lr=1e-2, print_each=1
)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)

# Train the model
trainer.fit(seq_model_complex, train_dataloaders=train_loader, val_dataloaders=val_loader)

## Test Sequences:

Test sequences are divided into three categories per environment:

* Reference (`native`)

* Alternative (`drift`)

* Paired (`paired`)

Use the `test_dataset_type`

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="native",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="native",transform=rev_transform,root="../data/",)

print("native info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="native")

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="drift",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="drift",transform=rev_transform,root="../data/",)

print("drift info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="drift")

### Note on paired sequences:

Each paired sequence contains: (1) A reference sequence (2) An alternative sequence (3) Differential expression column

In [30]:
dataset_paired = VaishnavDataset(
    split="test",
    dataset_env_type="complex",
    test_dataset_type="paired",
    root="../data/",
)
dataset_paired[0]

({'seq': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGGCTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT',
  'seq_alt': 'CTTTCAATTGGGTGGGGACGCGACGGCGCCCCGACTAGGATGCTAGCGTACTATGCTGCCTGAAAGTCTATAGGAGCATT'},
 tensor(-0.2812))

In [ ]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="paired",transform=forw_transform,root="../data/",)
test_rev = VaishnavDataset(split="test",dataset_env_type="complex",test_dataset_type="paired",transform=rev_transform,root="../data/",)

print("paired info")
print(test_forw)

forw_defined_native = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev_defined_native = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw_defined_native, rev_defined_native, trainer, seq_model_defined, name="paired")